# 🎯 [DAT06_M08] 수십 개 예하 부대 월간 결산 보고서 원클릭 일괄 생성 배치(Batch) 스트레스 테스트
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M08 |
| **미션 주제** | 수십 개 예하 부대 월간 결산 보고서 원클릭 일괄 생성 배치(Batch) 스트레스 테스트 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 수십 개 예하 부대 월간 결산 보고서 원클릭 일괄 생성 배치(Batch) 스트레스 테스트
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

수십 개 예하 부대 월간 결산 보고서 일괄 생성 배치 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **수십 개 예하 부대의 월간 결산 보고서를 원클릭으로 일괄 생성하는 배치를 만들고 장애·손상 자료를 주입해 스트레스 시험을 수행한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 부대 40개 x 12개월, 480건 보고서를 한 번에 생성 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/pipeline/dat06_batch_runner.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 수작업 대비 업무 소요 시간 80% 이상 절감 효과

#### 미션 PASS 조건

- 정량 기준: 전체 배치 파이프라인 실행 시간 5분 이내, 자동화 성공률 99% 이상


- 진행 기준: 스케줄러 연동 테스트 및 자동 실행 확인


- 검증 방법: 실제 월간 업무 데이터 1년치 일괄 배치 시뮬레이션


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""40개 가상 예하 부대 x 12개월(480건) 월간 결산 보고서를 한 번에 생성하는 배치를 실행하며, 일시 장애와 손상 자료를 주입해 재시도·격리 처리를 확인하고 전체 실행 시간, 자동화 성공률, 수작업 대비 절감률을 측정한다."""
import time
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

UNITS = [f"가상{i:02d}대대" for i in range(1, 41)]                 # 배치용으로 40개 부대로 확장
t_all = time.perf_counter()
rows = make_ledger(12, seed=8)
led = pd.DataFrame(rows)
lab = {f"2026-{i + 7:02d}": f"{2026 + (i + 6) // 12}-{(i + 6) % 12 + 1:02d}" for i in range(12)}   # 2026-07 ~ 2027-06
led["month"] = led["month"].map(lab)
months = sorted(led["month"].unique())
print(f"[배치 입력] 부대 {len(UNITS)}개 x {len(months)}개월 = {len(UNITS) * len(months)}건, 원장 {len(led)}행 [교육용가상자료]")

# 장애 주입: 일시 장애(재시도로 복구) 9건, 손상 자료(격리) 3건
rng = np.random.default_rng(80)
keys = [(u, m) for u in UNITS for m in months]
pick = rng.choice(len(keys), 12, replace=False)
transient = {keys[i] for i in pick[:9]}
corrupt = {keys[i] for i in pick[9:]}
for (u, m) in corrupt:                             # 기말 수치를 훼손(산식 불일치)
    i = led[(led["unit"] == u) & (led["month"] == m)].index[0]
    led.loc[i, "close"] = led.loc[i, "close"] + 999
attempts_log = {}


def build_report(u, m, g):
    k = (u, m)
    attempts_log[k] = attempts_log.get(k, 0) + 1
    if k in transient and attempts_log[k] == 1:
        raise TimeoutError("저장소 응답 지연(일시 장애)")
    bad = (g["open"] + g["recv"] + g["ret"] - g["issue"] != g["close"]).sum()
    if bad:
        raise ValueError(f"재고 산식 불일치 {int(bad)}건")
    s = g.groupby("cat")[["issue", "close"]].sum()
    return f"{u} {m} 불출 {int(s['issue'].sum())} 기말 {int(s['close'].sum())} (항목 {len(g)})"


done, quarantine, retries = {}, [], 0
t0 = time.perf_counter()
for (u, m), g in led.groupby(["unit", "month"]):
    for attempt in range(1, 4):                    # 최대 3회 재시도
        try:
            done[(u, m)] = build_report(u, m, g)
            break
        except TimeoutError:
            retries += 1
        except ValueError as ex:                   # 자료 오류는 재시도해도 소용없으므로 즉시 격리하고 다음 건으로 진행
            quarantine.append((u, m, str(ex)))
            break
run_s = time.perf_counter() - t0
total = len(keys)
succ = len(done) / total
print(f"\n[배치 실행] 생성 {len(done)}건, 격리 {len(quarantine)}건, 재시도 {retries}회")
for q in quarantine:
    print("  격리(담당자 확인 필요):", q)
print(f"보고서 생성 순수 실행 시간 {run_s:.2f}초, 전체(자료 생성 포함) {time.perf_counter() - t_all:.2f}초 (목표 5분 이내: {'달성' if time.perf_counter() - t_all < 300 else '미달'})")
print(f"자동화 성공률 {succ:.1%} (일시 장애 {len(transient)}건은 재시도로 자동 복구, 손상 자료는 격리 후 보고)")

# 수작업 대비 절감(가정값): 부대 보고서 1건 수작업 25분(미션 1 AS-IS 610분/24개 부대 기준), 자동화 후 사람 검토 2분/건, 격리 건 수동 처리 15분
manual = total * 25
auto = total * 2 + len(quarantine) * 15 + run_s / 60
print(f"\n[업무 소요 비교(가정값)] 수작업 {manual}분({manual / 60:.0f}시간) vs 자동화 {auto:.0f}분({auto / 60:.1f}시간) -> 절감 {1 - auto / manual:.1%}")

# 스트레스: 부하 3배(1,440건) 반복 실행의 처리량
big = list(led.groupby(["unit", "month"]))
t0 = time.perf_counter(); n = 0
for _ in range(3):
    for (u, m), g in big:
        if (u, m) in corrupt:
            continue
        attempts_log[(u, m)] = 9                     # 일시 장애 주입 해제(처리량 측정 목적)
        build_report(u, m, g); n += 1
dt = time.perf_counter() - t0
print(f"부하 시험 {n}건 처리 {dt:.2f}초 (초당 {n / dt:.0f}건), 오류 0건")
print(f"\n목표(전체 5분 이내, 자동화 성공률 99% 이상, 수작업 대비 80% 이상 절감): "
      f"{'달성' if run_s < 300 and succ >= 0.99 and 1 - auto / manual >= 0.8 else '미달'}")


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[배치 입력] 부대 40개 x 12개월 = 480건, 원장 5280행 [교육용가상자료]

[배치 실행] 생성 477건, 격리 3건, 재시도 9회
  격리(담당자 확인 필요): ('가상10대대', '2027-05', '재고 산식 불일치 1건')
  격리(담당자 확인 필요): ('가상24대대', '2027-02', '재고 산식 불일치 1건')
  격리(담당자 확인 필요): ('가상36대대', '2026-12', '재고 산식 불일치 1건')
보고서 생성 순수 실행 시간 1.06초, 전체(자료 생성 포함) 1.13초 (목표 5분 이내: 달성)
자동화 성공률 99.4% (일시 장애 9건은 재시도로 자동 복구, 손상 자료는 격리 후 보고)

[업무 소요 비교(가정값)] 수작업 12000분(200시간) vs 자동화 1005분(16.8시간) -> 절감 91.6%
부하 시험 1431건 처리 3.63초 (초당 394건), 오류 0건

목표(전체 5분 이내, 자동화 성공률 99% 이상, 수작업 대비 80% 이상 절감): 달성
```
</details>
